## Data preparation summary

This notebook prepares the SBA loan file for downstream featurization by normalizing the raw schema, deriving a status target, and retaining only loans with an observed terminal outcome. It retains all cleaned columns; feature selection and leakage controls are handled in the featurization pipeline.

The workflow is limited to:
- normalize column names and types to the project convention
- derive `loan_status_r` as `-1 = being serviced`, `0 = paid in full`, and `1 = charged off`
- keep `PIF` and `CHGOFF` for model development; exclude `EXEMPT`, `COMMIT`, and `CANCLD`
- split the terminal-outcome universe into stratified train, validation, and test sets
- compute `loan_age_months` from `AsOfDate` using `FirstDisbursementDate` when available, otherwise `ApprovalDate`
- retain every cleaned source column plus the derived fields


## Data preparation notes

Status definitions come from the SBA 7(a) / 504 data dictionary in `data_dictionary/sba_dd.csv`, not from the origination SOP. The raw labels are authoritative for this file:

- **Paid in full (`PIF`, shown as `P I F` in the source):** the loan has been paid in full.
- **Charged off (`CHGOFF`):** SBA charged off the loan.
- **Being serviced (`EXEMPT`):** the loan has been disbursed and has not been cancelled, paid in full, or charged off. This is the dataset's proxy for a loan remaining in the servicing/repayment process; it does not by itself establish that the borrower is current, delinquent, or actively making a payment on the snapshot date.

SOP 50 10 8, page 9, confirms that post-disbursement loans fall under separate servicing procedures (SOP 50 57 for 7(a) and SOP 50 55 for 504). It does not define these dataset labels. `EXEMPT` is therefore inferred directly from the supplied data dictionary's definition, rather than from SOP 50 10.

The current terminal-outcome universe contains **75,094 rows**: 68,201 paid in full and 6,893 charged off. `EXEMPT` loans are not terminal: their eventual outcome is not yet known, so including them as successful or unsuccessful would create right-censored labels and prevent unbiased model evaluation. Active loans can still be scored by the trained model later, but they do not belong in training, validation, or test.


## Dataset contract

The downstream dataset contract is:
- raw `LoanStatus` values retained: `PIF`/`P I F` and `CHGOFF`
- recoded `loan_status_r` values in the development universe: `0 = paid in full`, `1 = charged off`
- raw statuses `EXEMPT` (active/being serviced), `COMMIT` (undisbursed), and `CANCLD` (cancelled) excluded
- `loan_age_months` calculated from `AsOfDate` and the earliest valid loan start date
- all cleaned source columns retained for later featurization decisions
- stratified 80%/10%/10% train/validation/test split using random seed 42
- split files exported to `documents/dd_cleaner/sba_user_cleaned_raw_training.csv`, `sba_user_cleaned_raw_validation.csv`, and `sba_user_cleaned_raw_test.csv`


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

from dd_cleaner.notebook_utils import (
    init_notebook_session, 
    get_raw_data, 
    get_cleaned_data, 
    get_tagged_entities,
    get_attributes_by_tag
)

# Initialize the session using the actual workspace root.
# When running from the notebooks/ directory, the project root sits one level up.
workspace_root = Path.cwd().resolve()
if not (workspace_root / "config.yaml").exists():
    workspace_root = workspace_root.parent

coord, artifacts_df = init_notebook_session(str(workspace_root))

# Display the artifacts table
artifacts_df

# 1. Initialize session and load the 'Clean Baseline'
# Point to the directory containing the 'data/' folder
coord, _ = init_notebook_session(str(workspace_root))
df = get_cleaned_data(coord)

print(f"Loaded Baseline: {df.shape[0]} rows, {df.shape[1]} columns")


✅ Notebook session initialized for workspace: /home/rajiv/programming/kmds_migration/sba_migration

Available Artifacts:

Artifact Name                                  File Name  \
0                         Raw Data                          sba_loans_raw.csv   
1                     Cleaned Data                    sba_loans_raw_clean.csv   
2                User Cleaned Data                 sba_loans_user_cleaned.csv   
3             Tagged Entities (DD)         sba_loans_raw_analysis_results.csv   
4  Cleaning Recommendations Report                cleaning_recommendations.md   
5                 Profiling Report          sba_loans_raw_profiling_report.md   
6                   Handshake File  sba_loans_raw_parser_cleaner_handshake.md   
7                  Quarantine File               sba_loans_raw_quarantine.csv   
8               Metadata Authority           sba_loans_raw_metadata_table.csv   

                                            Location  Exists  
0                             data/sba_loans_raw.csv    True  
1            data/dd_cleaner/sba_loans_raw_clean.csv    True  
2         data/dd_cleaner/sba_loans_user_cleaned.csv    True  
3  documents/dd_analysis_results/sba_loans_raw_an...    True  
4   documents/dd_cleaner/cleaning_recommendations.md    True  
5  documents/dd_cleaner/sba_loans_raw_profiling_r...    True  
6  documents/dd_cleaner/sba_loans_raw_parser_clea...    True  
7       data/quarantine/sba_loans_raw_quarantine.csv   False  
8   data/dd_cleaner/sba_loans_raw_metadata_table.csv    True

✅ Notebook session initialized for workspace: /home/rajiv/programming/kmds_migration/sba_migration

Available Artifacts:

Artifact Name                                  File Name  \
0                         Raw Data                          sba_loans_raw.csv   
1                     Cleaned Data                    sba_loans_raw_clean.csv   
2                User Cleaned Data                 sba_loans_user_cleaned.csv   
3             Tagged Entities (DD)         sba_loans_raw_analysis_results.csv   
4  Cleaning Recommendations Report                cleaning_recommendations.md   
5                 Profiling Report          sba_loans_raw_profiling_report.md   
6                   Handshake File  sba_loans_raw_parser_cleaner_handshake.md   
7                  Quarantine File               sba_loans_raw_quarantine.csv   
8               Metadata Authority           sba_loans_raw_metadata_table.csv   

                                            Location  Exists  
0                             data/sba_loans_raw.csv    True  
1            data/dd_cleaner/sba_loans_raw_clean.csv    True  
2         data/dd_cleaner/sba_loans_user_cleaned.csv    True  
3  documents/dd_analysis_results/sba_loans_raw_an...    True  
4   documents/dd_cleaner/cleaning_recommendations.md    True  
5  documents/dd_cleaner/sba_loans_raw_profiling_r...    True  
6  documents/dd_cleaner/sba_loans_raw_parser_clea...    True  
7       data/quarantine/sba_loans_raw_quarantine.csv   False  
8   data/dd_cleaner/sba_loans_raw_metadata_table.csv    True

Loaded Baseline: 388338 rows, 42 columns


/home/rajiv/programming/kmds_migration/sba_migration/.venv/lib/python3.13/site-packages/dd_cleaner/notebook_utils.py:124: DtypeWarning: Columns (0: ChargeOffDate) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path)


In [2]:
retained_statuses = {
    "PIF": ("Paid in full", 0),
    "CHGOFF": ("Charged off", 1),
}
normalized_loan_status = (
    df["LoanStatus"]
    .astype("string")
    .str.upper()
    .str.replace(r"[\s\-_/()]+", "", regex=True)
)
status_accounting = (
    normalized_loan_status
    .value_counts(dropna=False)
    .rename_axis("Raw Status")
    .reset_index(name="Count")
)
status_accounting["Definition"] = status_accounting["Raw Status"].map(
    {key: value[0] for key, value in retained_statuses.items()}
).fillna("Excluded")
status_accounting["loan_status_r"] = status_accounting["Raw Status"].map(
    {key: value[1] for key, value in retained_statuses.items()}
).astype("Int64")
status_accounting["Disposition"] = status_accounting["Raw Status"].map(
    lambda value: "Retain" if value in retained_statuses else "Exclude"
)
status_accounting["Percentage of all rows"] = (
    status_accounting["Count"] / len(df)
)
status_accounting.style.format({
    "Count": "{:,}",
    "Percentage of all rows": "{:.2%}",
}).hide(axis="index")

Raw Status,Count,Definition,loan_status_r,Disposition,Percentage of all rows
EXEMPT,"242,061",Excluded,,Exclude,62.33%
PIF,"68,201",Paid in full,0,Retain,17.56%
CANCLD,"50,104",Excluded,,Exclude,12.90%
COMMIT,"21,079",Excluded,,Exclude,5.43%
CHGOFF,"6,893",Charged off,1,Retain,1.78%


## The Tagged Dictionary
This is the programatic component that has the gathered intelligence and can be used to apply cleaning rules based on abstract rules we specify in the configuration stage.

In [3]:
# Load the metadata/tagged entities
df_tags = get_tagged_entities(coord)

In [4]:
df_tags.head()

,Field Name,Definition,physical_type,logical_type,attribute_name,provisional_entity_assignment,static_dynamic,is_geographical
0,AsOfDate,Date when the data was recorded,datetime,datetime,AsOfDate,Logical Category,none,False
1,Program,Indicator of whether loan was approved under S...,datetime,datetime,Program,Logical Categories,none,False
2,LocationID,SBA's unique lender ID code,int,numeric,LocationID,Logical Category,none,False
3,BorrName,Borrower name,str,text,BorrName,Person,none,False
4,BorrStreet,Borrower street address,str,text,BorrStreet,Logical Categories,none,False


In [5]:
from pathlib import Path
import re

# Resolve the cleaning recommendations file from the workspace root or the notebook folder.
workspace_root = Path.cwd().resolve()
if not (workspace_root / "config.yaml").exists():
    workspace_root = workspace_root.parent

candidate_paths = [
    workspace_root / "documents" / "dd_cleaner" / "cleaning_recommendations.md",
    workspace_root / ".." / "documents" / "dd_cleaner" / "cleaning_recommendations.md",
]
fp_cleaning_recs = next((p for p in candidate_paths if p.exists()), candidate_paths[0])
if not fp_cleaning_recs.exists():
    raise FileNotFoundError(f"Cleaning recommendations file not found. Looked in: {candidate_paths}")

markdown_text = fp_cleaning_recs.read_text(encoding="utf-8")

all_columns = df.columns.tolist()
categorical_columns = df_tags[df_tags['logical_type'] == 'categorical']['attribute_name'].tolist()
numerical_columns = df_tags[df_tags['logical_type'] == 'numerical']['attribute_name'].tolist()
datetime_columns = df_tags[df_tags['logical_type'] == 'datetime']['attribute_name'].tolist()

def markdown_table_to_df(table_markdown: str) -> pd.DataFrame:
    """Convert a single markdown pipe table block to a DataFrame."""
    lines = [ln.strip() for ln in table_markdown.strip().splitlines() if ln.strip().startswith("|")]
    # Drop the markdown alignment row like |:---|:---|
    lines = [ln for ln in lines if not re.match(r"^\|[:\-\s|]+\|?$", ln)]

    rows = [[cell.strip() for cell in ln.strip("|").split("|")] for ln in lines]
    header, data = rows[0], rows[1:]
    return pd.DataFrame(data, columns=header)

# Extract all contiguous markdown table blocks and combine them
table_blocks = re.findall(r"(?:^\|.*\|\n)+", markdown_text, flags=re.MULTILINE)
df_cleaning_recs = pd.concat(
    [markdown_table_to_df(block) for block in table_blocks],
    ignore_index=True
)

df_cleaning_recs


,Attribute,Type,Entity,What Needs Fixing,Recommended Fix
0,AsOfDate,datetime,Logical Category,Constant value / Zero variance,drop-attribute
1,Program,datetime,Logical Categories,Constant value / Zero variance,drop-attribute
2,BankNCUANumber,numeric,Logical Categories,Extreme sparsity (97.1%): Exceeds null thresho...,drop-attribute
3,ChargeOffDate,datetime,Logical Categories,Extreme sparsity (98.2%): Exceeds null thresho...,drop-attribute
4,SoldSecMrktInd,categorical,Logical Categories,Constant value / Zero variance,drop-attribute
5,ApprovalDate,datetime,Logical Categories,This is a cross sectional dataset; if you want...,custom:datetime_to_numeric
6,FirstDisbursementDate,datetime,Logical Categories,This is a cross sectional dataset; if you want...,custom:datetime_to_numeric
7,PaidInFullDate,datetime,Logical Category,This is a cross sectional dataset; if you want...,custom:datetime_to_numeric
8,BankFDICNumber,numeric,Logical Categories,Numeric with 10.6% nulls: Recommendation is me...,mean-imputation
9,InitialInterestRate,numeric,Logical Categories,Numeric with 0.0% nulls: Recommendation is mea...,mean-imputation


### Column-retention boundary

This notebook performs status-based row selection and normalization only. It retains every cleaned source column. Potential outcome-leakage fields, high-cardinality identifiers, location fields, dates, and other feature-engineering decisions remain available for explicit treatment in the featurization pipeline.


### Final Status Contract

The cleaned dataset retains the original columns and adds one recoded target, `loan_status_r`.

- `loan_status_r == 0`: paid in full
- `loan_status_r == 1`: charged off

The `LoanStatus` definitions are taken from `data_dictionary/sba_dd.csv`. `EXEMPT` is excluded because it represents a disbursed loan without a terminal outcome, so its eventual good/bad result is not yet observable. `COMMIT` is excluded because it is undisbursed, and `CANCLD` is excluded because it is cancelled. These exclusions define an evaluable terminal-outcome universe; they are not assumptions about feature relevance.

The `loan_age_months` field is derived from `AsOfDate` using the earliest valid loan-start signal (`FirstDisbursementDate` when available, otherwise `ApprovalDate`). This keeps the seasoning feature aligned to the observed loan age at the snapshot date.


In [6]:
# The raw status remains in the cleaned frame for auditability. The derived target
# is a separate field so the featurization pipeline can choose how to consume it.
loan_status_target_name = "loan_status_r"

def normalize_column_name(name: object) -> str:
    return re.sub(r"[\s_]+", "", str(name)).lower()

loan_status_target_name


'loan_status_r'

In [7]:
# Normalize the dataframe contract so raw and cleaned files with different casing
# and spacing still work with the notebook's expected lowercase field names.
def canonicalize_dataframe_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy of the dataframe with normalized lowercase column names."""
    def normalize_name(name: object) -> str:
        return re.sub(r"[\s_]+", "", str(name)).lower()
    return df.rename(columns=lambda name: normalize_name(name))


def derive_loan_status_r(df: pd.DataFrame) -> pd.DataFrame:
    """Map SBA servicing/outcome statuses to serviced=-1, PIF=0, charge-off=1."""
    work_df = canonicalize_dataframe_columns(df)
    loanstatus = work_df["loanstatus"].astype("string").str.upper().str.strip()
    loanstatus_key = loanstatus.str.replace(r"[\s\-_/()]+", "", regex=True)

    status_mapping = {
        "PIF": 0,
        "CHGOFF": 1,
    }

    work_df[loan_status_target_name] = loanstatus_key.map(status_mapping).astype("Int64")
    return work_df


def derive_loan_age_months(df: pd.DataFrame) -> pd.DataFrame:
    """Compute loan age in months from the snapshot date using the loan start date."""
    work_df = df.copy()
    first_disbursement = pd.to_datetime(work_df.get("firstdisbursementdate", pd.NaT), errors="coerce")
    approval_date = pd.to_datetime(work_df.get("approvaldate", pd.NaT), errors="coerce")
    asof_date = pd.to_datetime(work_df.get("asofdate", pd.NaT), errors="coerce")

    loan_start_date = first_disbursement.combine_first(approval_date)
    work_df["loan_age_months"] = (asof_date - loan_start_date).dt.days / 30.4375
    return work_df


In [8]:

# Load the metadata/tagged entities
df_tags = get_tagged_entities(coord)

In [9]:
df_tags

,Field Name,Definition,physical_type,logical_type,attribute_name,provisional_entity_assignment,static_dynamic,is_geographical
0,AsOfDate,Date when the data was recorded,datetime,datetime,AsOfDate,Logical Category,none,False
1,Program,Indicator of whether loan was approved under S...,datetime,datetime,Program,Logical Categories,none,False
2,LocationID,SBA's unique lender ID code,int,numeric,LocationID,Logical Category,none,False
3,BorrName,Borrower name,str,text,BorrName,Person,none,False
4,BorrStreet,Borrower street address,str,text,BorrStreet,Logical Categories,none,False
5,BorrCity,Borrower city,str,text,BorrCity,Location,none,True
6,BorrState,Borrower state,str,categorical,BorrState,Logical Category,none,False
7,BorrZip,Borrower zip code,int,numeric,BorrZip,Geographical,none,False
8,BankName,Name of the bank that the loan is currently as...,str,categorical,BankName,Logical Category,none,False
9,BankFDICNumber,The Federal Depository Insurance Corporation c...,float,numeric,BankFDICNumber,Logical Categories,none,False


In [10]:
cat_attrib_filter = df_tags.logical_type == "categorical"
cat_attrib = df_tags[cat_attrib_filter]["Field Name"]

In [11]:
datetime_attrib_filter = df_tags.logical_type == "datetime"
numeric_attrib_filter = df_tags.logical_type == "numeric"
date_time_attrib =  df_tags[datetime_attrib_filter]["Field Name"]
numeric_attrib = df_tags[numeric_attrib_filter]["Field Name"]

In [12]:
def rectify_types(df: pd.DataFrame) -> pd.DataFrame:
    """Coerce selected fields to string-backed categorical in both data and df_tags."""
    global df_tags

    target_columns = {"naicscode", "franchisecode", "congressionaldistrict", "borrzip"}
    normalized_target_columns = {col.replace(" ", "").replace("_", "").lower() for col in target_columns}

    work_df = df.copy()
    normalized_df_columns = {
        column: column.replace(" ", "").replace("_", "").lower()
        for column in work_df.columns
    }

    for column, normalized_column in normalized_df_columns.items():
        if normalized_column in normalized_target_columns:
            work_df[column] = work_df[column].astype("string").astype("category")

    if "attribute_name" in df_tags.columns:
        normalized_attribute_names = (
            df_tags["attribute_name"]
            .astype("string")
            .str.replace(" ", "", regex=False)
            .str.replace("_", "", regex=False)
            .str.lower()
        )
        target_mask = normalized_attribute_names.isin(normalized_target_columns)

        if "logical_type" in df_tags.columns:
            df_tags.loc[target_mask, "logical_type"] = "categorical"

        if "physical_type" in df_tags.columns:
            df_tags.loc[target_mask, "physical_type"] = "str"

    return work_df

In [13]:
def handle_missing_categorical(df: pd.DataFrame) -> pd.DataFrame:
    """Fill missing values in categorical fields with the literal category 'MISSING'."""
    global df_tags

    work_df = df.copy()
    normalized_df_columns = {
        column: column.replace(" ", "").replace("_", "").lower()
        for column in work_df.columns
    }

    categorical_targets = set()
    if "attribute_name" in df_tags.columns and "logical_type" in df_tags.columns:
        categorical_targets = set(
            df_tags.loc[
                df_tags["logical_type"].astype("string").str.lower() == "categorical",
                "attribute_name",
            ]
            .astype("string")
            .str.replace(" ", "", regex=False)
            .str.replace("_", "", regex=False)
            .str.lower()
        )

    for column, normalized_column in normalized_df_columns.items():
        if normalized_column in categorical_targets:
            if isinstance(work_df[column].dtype, pd.CategoricalDtype):
                if "MISSING" not in work_df[column].cat.categories:
                    work_df[column] = work_df[column].cat.add_categories(["MISSING"])
                work_df[column] = work_df[column].fillna("MISSING")
            else:
                work_df[column] = work_df[column].astype("string").fillna("MISSING").astype("category")

    return work_df


def fill_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """Fill any remaining numeric, datetime, or categorical missing values before saving the cleaned data."""
    work_df = df.copy()
    for column in work_df.columns:
        series = work_df[column]
        if series.isna().sum() == 0:
            continue
        if pd.api.types.is_numeric_dtype(series):
            work_df[column] = series.fillna(series.median())
        elif pd.api.types.is_datetime64_any_dtype(series):
            work_df[column] = series.fillna(pd.Timestamp("1970-01-01"))
        else:
            work_df[column] = series.astype("string").fillna("MISSING")
    return work_df


In [14]:
def filter_valid_loancondition(df: pd.DataFrame) -> pd.DataFrame:
    """Keep only terminal outcomes that can support model evaluation."""
    work_df = df.copy()
    if loan_status_target_name not in work_df.columns:
        work_df = derive_loan_status_r(work_df)
    return work_df.loc[work_df[loan_status_target_name].isin([0, 1])].copy()


Note: `estimated_balance` is an approximate remaining unpaid balance proxy for decisioning only. It is not a true accounting value or borrower-level repayment ledger. The raw SBA dataset does not include the amortization schedule, payment history, or actual outstanding principal, so this feature is intended as a rough exposure estimate until those data become available.

The calculation uses the loan amount, term, interest rate, and elapsed loan age to estimate a remaining balance under a simple fixed-rate amortization assumption.


In [15]:
# Execute the status-cleaning chain.
df_raw_baseline = get_cleaned_data(coord)

df_clean = canonicalize_dataframe_columns(df_raw_baseline)
df_clean = derive_loan_status_r(df_clean)
df_clean = filter_valid_loancondition(df_clean)
df_clean = derive_loan_age_months(df_clean)
df_clean = rectify_types(df_clean)
df_clean = handle_missing_categorical(df_clean)
df_clean = fill_missing_values(df_clean)

train_df, holdout_df = train_test_split(
    df_clean,
    test_size=0.2,
    stratify=df_clean[loan_status_target_name],
    random_state=42,
)
validation_df, test_df = train_test_split(
    holdout_df,
    test_size=0.5,
    stratify=holdout_df[loan_status_target_name],
    random_state=42,
)

split_output_dir = workspace_root / "documents" / "dd_cleaner"
split_output_dir.mkdir(parents=True, exist_ok=True)
split_files = {
    "training": train_df,
    "validation": validation_df,
    "test": test_df,
}
assert set(train_df.index).isdisjoint(validation_df.index)
assert set(train_df.index).isdisjoint(test_df.index)
assert set(validation_df.index).isdisjoint(test_df.index)
assert len(train_df) + len(validation_df) + len(test_df) == len(df_clean)
for split_name, split_df in split_files.items():
    split_path = split_output_dir / f"sba_user_cleaned_raw_{split_name}.csv"
    split_df.to_csv(split_path, index=False)
    print(f"Saved {split_name} dataset: {split_path} ({split_df.shape})")
    print(split_df[loan_status_target_name].value_counts().sort_index().to_dict())

print(f"Terminal-outcome universe shape: {df_clean.shape}")


/home/rajiv/programming/kmds_migration/sba_migration/.venv/lib/python3.13/site-packages/dd_cleaner/notebook_utils.py:124: DtypeWarning: Columns (0: ChargeOffDate) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path)


Saved training dataset: /home/rajiv/programming/kmds_migration/sba_migration/documents/dd_cleaner/sba_user_cleaned_raw_training.csv ((60075, 44))
{np.int64(0): 54561, np.int64(1): 5514}
Saved validation dataset: /home/rajiv/programming/kmds_migration/sba_migration/documents/dd_cleaner/sba_user_cleaned_raw_validation.csv ((7509, 44))
{np.int64(0): 6820, np.int64(1): 689}
Saved test dataset: /home/rajiv/programming/kmds_migration/sba_migration/documents/dd_cleaner/sba_user_cleaned_raw_test.csv ((7510, 44))
{np.int64(0): 6820, np.int64(1): 690}
Terminal-outcome universe shape: (75094, 44)


In [16]:
missing_values_report = (
    df_clean.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
    .assign(
        missing_pct=lambda d: (d["missing_count"] / len(df_clean) * 100).round(2)
    )
    .query("missing_count > 0")
    .sort_values("missing_count", ascending=False)
    .reset_index(names="column")
)

print(f"Columns with unhandled missing values: {missing_values_report.shape[0]}")
missing_values_report


Columns with unhandled missing values: 0


,column,missing_count,missing_pct


In [17]:
labels = df_clean[loan_status_target_name].value_counts(dropna=False)
print(labels.to_string())


loan_status_r
0    68201
1     6893


In [18]:
counts = df_clean[loan_status_target_name].value_counts(dropna=False)
charged_off_count = counts.get(1, 0)
paid_in_full_count = counts.get(0, 0)
labeled_count = charged_off_count + paid_in_full_count

imbalance_ratio = charged_off_count / paid_in_full_count if paid_in_full_count else np.nan
bad_share_of_labeled = charged_off_count / labeled_count if labeled_count else np.nan

print(f"Imbalance ratio (charged-off/paid-in-full): {imbalance_ratio:.6f}")
print(f"Bad share among labeled rows (charged-off / (charged-off + paid-in-full)): {bad_share_of_labeled:.6f}")
print(f"Bad share among labeled rows (percent): {bad_share_of_labeled * 100:.2f}%")

imbalance_ratio, bad_share_of_labeled


Imbalance ratio (charged-off/paid-in-full): 0.101069
Bad share among labeled rows (charged-off / (charged-off + paid-in-full)): 0.091792
Bad share among labeled rows (percent): 9.18%


(np.float64(0.10106889928300171), np.float64(0.09179162116813594))

In [19]:
df_clean.columns


Index(['asofdate', 'program', 'locationid', 'borrname', 'borrstreet',
       'borrcity', 'borrstate', 'borrzip', 'bankname', 'bankfdicnumber',
       'bankncuanumber', 'bankstreet', 'bankcity', 'bankstate', 'bankzip',
       'grossapproval', 'sbaguaranteedapproval', 'approvaldate', 'approvalfy',
       'firstdisbursementdate', 'processingmethod', 'initialinterestrate',
       'fixedorvariableinterestind', 'terminmonths', 'naicscode',
       'naicsdescription', 'franchisecode', 'franchisename', 'projectcounty',
       'projectstate', 'sbadistrictoffice', 'congressionaldistrict',
       'businesstype', 'businessage', 'loanstatus', 'paidinfulldate',
       'chargeoffdate', 'grosschargeoffamount', 'revolverstatus',
       'jobssupported', 'collateralind', 'soldsecmrktind', 'loan_status_r',
       'loan_age_months'],
      dtype='str')

### Save the metadata to the correct location

In [20]:
from dd_cleaner.notebook_utils import get_metadata_table, save_metadata_table
save_metadata_table(coord, df_tags)

✅ Metadata Authority (Expert Overrides) saved to: /home/rajiv/programming/kmds_migration/sba_migration/data/dd_cleaner/sba_loans_raw_metadata_table.csv


## Final summary

This notebook now produces and exports a terminal-outcome development universe while preserving the full cleaned schema. Active/undetermined loans are intentionally excluded from model evaluation because they do not yet have an observed outcome. Feature selection remains the responsibility of the featurization pipeline.

### What the migration now does end-to-end

1. Reads the raw SBA data and normalizes the schema.
2. Derives `loan_status_r` from the data-dictionary labels: `0 = paid in full (PIF)` and `1 = charged off (CHGOFF)`.
3. Excludes `EXEMPT` because no terminal outcome is known, `COMMIT` because it is undisbursed, and `CANCLD` because it is cancelled.
4. Calculates `loan_age_months` from `AsOfDate` using `FirstDisbursementDate` when available, otherwise `ApprovalDate`.
5. Retains all cleaned source columns for downstream feature analysis and leakage handling.
6. Stratifies the terminal-outcome universe into 80% train, 10% validation, and 10% test with random seed 42.
7. Writes `sba_user_cleaned_raw_training.csv`, `sba_user_cleaned_raw_validation.csv`, and `sba_user_cleaned_raw_test.csv` under `documents/dd_cleaner/`.

### Notes for the analyst

- The SBA can later apply the trained model to active loans, but their eventual outcomes are unavailable for unbiased train/validation/test evaluation in this dataset.
- Training is used to fit the model, validation to tune performance decisions, and test only for final performance evaluation.
- `loan_status_r` remains available separately from `loanstatus` for downstream consumers.
- No model-feature selection is performed in this notebook.
